## QAOA for Max Cut optimization problem

In [ ]:
from qarp.algorithms import QAOA

import networkx as nx
import matplotlib.pyplot as plt
import itertools

### Random graph generation

We use networkx functionalities to generate a random graph. In the interest of visualization and comprehension, here we generate a graph of 4 nodes and just 2 edges. This is parameterized for a general purpose.

In [ ]:
from qarp import config

config.seed = 1234

In [ ]:
from qarp.graphs import Graph

n_nodes = 4
n_edges = 2

G = nx.gnm_random_graph(n_nodes, n_edges, seed=config.seed)
for u, v in G.edges:
    G[u][v]['weight'] = 2

graph = Graph(G)
graph.plot()

### QAOA algorithm

OpenQARP implements the QAOA in a general way. It takes a graph over which the edge cut maximization is run. Additionally, the number of sequential layers in the ansatz and the set of initial parameters are parameters of the algorithm. Please, check the documentation for a more detailed customization of the approach. 

Note that the current API also allows to take a Hamiltonian instead of a graph. See the example later.

The engine and type of primitive can also be customizable in the QAOA approach.

Since the QAOA circuit can be built by using CZ or CX, this is customizable in the QAOA arguments.

In [ ]:
n_layers = 2
qaoa = QAOA(problem=G, n_layers=n_layers, use_rzz=True, verbose=True, initial_parameters=[0]*2*n_layers).build()

The following cell shows the QAOA ansatz

In [ ]:
qaoa.ket.plot()


The following line runs the QAOA approach with the default optimizer

In [ ]:
fun, x = qaoa.run()

The QAOA approach is finding the solutions in which the cost function is around fun=4, which means that is cutting two different edges in the original graph.

In [ ]:
print("Expectation value:", fun)
print("Optimal parameters:", x)

Let's measure the circuit with the optimal parameters and shos a histogram with the optimal solutions

In [ ]:
from qarp.engines import QarpEngine
from qarp.algorithms import Sampler
from copy import deepcopy
# run() already binds result.x onto the sorted symbol tuple (§17).
circ = deepcopy(qaoa.get_final_state_block())
circ.measure([(q, q) for q in range(circ.n_qubits)])

engine = QarpEngine()
sampler = Sampler(ket=circ, n_shots=10000)
engine.build([sampler])
engine.run({})
probs = sampler.result


The following solutions have equal probability of occurence. Since there are several optimum solutions, some of them have equal probability. 

In this case, we have two edges from 0 to other two nodes {2, 3}. If a subset includes 0 and the other subset includes 2 and 3, then we are cutting all the edges in the graph. The following solutions represent this situation.

Note that, for example, (0, 0, 1, 1) and (1, 1, 0, 0) represent the same solution.

In [ ]:
probs

In [ ]:
from qarp.plotting import plot_histogram
plot_histogram(probs, show_all_solutions=True)

## Hamiltonian example

In this example we define a Hamiltonian to be solved, instead of using a graph. Note that by using this new API we are able to add linear terms to the Hamiltonian so that we can add constraints to the problem

We define here a problem of three variables

In [ ]:
from qarp.operators import QubitOperator

ham = QubitOperator("Z0 Z1") + QubitOperator("Z1 Z2")  # (0, 1, 0) or (1, 0, 1)
ham += QubitOperator("Z0")  # penalize (0, 1, 0)

print(ham)

In the interest of showing different QAOA features, here we use gradients during the optimization process.

In [ ]:
from qarp.algorithms import TermwiseHadamardTest

qaoa = QAOA(ham, n_layers=4, verbose=True, gradient=False).build()
fun, x = qaoa.run()

In [ ]:
from qarp.engines import QarpEngine
from qarp.algorithms import Sampler
from copy import deepcopy
# run() already binds result.x onto the sorted symbol tuple (§17).
circ = deepcopy(qaoa.get_final_state_block())
circ.measure([(q, q) for q in range(circ.n_qubits)])

engine = QarpEngine()
sampler = Sampler(ket=circ, n_shots=10000)
engine.build([sampler])
engine.run({})
probs = sampler.result


In [ ]:
probs

In [ ]:
from qarp.plotting import plot_histogram
plot_histogram(probs, show_all_solutions=True)

We can see that the solution (1, 0, 1) is the most probable since it is the one enhanced by the linear terms.